# 06 — Alpha System
Cross-sectional Ridge/LightGBM, walk-forward CV, OOS portfolio construction.

In [1]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('../src'))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Exact function signatures verified against src/:
#   simulate_sentiment(dates) -> pd.DataFrame  (has 'sentiment_1d' column)
#   detect_events(returns, sentiment_series, ...) -> pd.Series (boolean)
#   build_panel(returns, sentiment_series, events, target_horizon, stocks) -> pd.DataFrame
#   AlphaModel(model_type).fit(panel) -> AlphaModel
#   AlphaModel.predict(panel) -> pd.Series
#   AlphaModel.top_features(n) -> pd.Series
#   AlphaModel.save(name) / AlphaModel.load(name)
#   AlphaModel.predict_latest(live_panel) -> pd.Series
#   time_split(panel, train_ratio) -> (train_df, test_df)
#   build_portfolio(predicted_returns, actual_returns, top_k, bottom_k, hold_days, tc)
#   daily_pnl(positions) -> pd.Series
#   portfolio_summary(positions) -> Dict
#   compute_metrics(pnl_series) -> Dict
#   compute_ic(predictions, actuals) -> pd.Series
#   nifty_benchmark(returns, event_dates) -> Dict
#   build_live_features(returns, sentiment_series, stocks, as_of) -> pd.DataFrame
#   rank_signals(predicted_returns) -> pd.Series
#   signal_weights(signals) -> pd.Series

from data import load_returns
from sentiment import simulate_sentiment
from events import detect_events
from features import build_panel, build_live_features, FEATURE_COLS
from model import AlphaModel, time_split
from strategy import build_portfolio, daily_pnl, portfolio_summary, rank_signals, signal_weights
from backtest import compute_metrics, compute_ic, nifty_benchmark
from utils import DATA_PROC, TABLES_DIR, PLOTS_DIR, set_theme, save_table
from constants import MARKET_COL, OIL_COL, FX_COL, STOCK_UNIVERSE


In [2]:
# Load cached stock-level returns (no internet required)
returns = load_returns()
print(f'Returns : {returns.shape}')
print(f'Columns : {list(returns.columns)}')


Returns : (1928, 18)
Columns : ['RELIANCE', 'ONGC', 'IOC', 'BPCL', 'INDIGO', 'HPCL', 'ADANIPORTS', 'TATAMOTORS', 'MARUTI', 'ASIANPAINT', 'HINDUNILVR', 'ITC', 'TCS', 'INFY', 'CIPLA', 'BRENT', 'NIFTY', 'USDINR']


In [3]:
# Simulated sentiment — simulate_sentiment() returns a DataFrame.
# build_panel / detect_events expect a Series (sentiment_1d column).
sent_df = simulate_sentiment(returns.index)       # pd.DataFrame
sent    = sent_df['sentiment_1d']                 # pd.Series  ← correct arg type
print(f'Sentiment DataFrame: {sent_df.shape}')
print(f'Sentiment Series   : mean={sent.mean():.4f}  non-zero={sent.ne(0).sum()}')


Sentiment DataFrame: (1928, 5)
Sentiment Series   : mean=0.0016  non-zero=1928


In [4]:
# Event detection sensitivity table
modes      = ['oil_only', 'oil_and_sentiment', 'oil_or_sentiment']
thresholds = [0.03, 0.04, 0.05]
rows = []
for thr in thresholds:
    for mode in modes:
        n = detect_events(returns, sent, threshold=thr, mode=mode).sum()
        rows.append({'threshold': f'{thr:.0%}', 'mode': mode, 'n_events': int(n)})

ev_sens = pd.DataFrame(rows).pivot(index='threshold', columns='mode', values='n_events')
print(ev_sens.to_string())


mode       oil_and_sentiment  oil_only  oil_or_sentiment
threshold                                               
3%                        19       243               250
4%                        14       181               196
5%                        10       113               129


In [5]:
# Build cross-sectional panel (stocks × event-dates × features)
events = detect_events(returns, sent, mode='oil_only')
print(f'Event days: {events.sum()}')

panel = build_panel(returns, sent, events, target_horizon=3)
print(f'Panel shape   : {panel.shape}')
print(f'Event dates   : {panel.index.get_level_values("date").nunique()}')
print(f'Stocks        : {panel.index.get_level_values("stock").nunique()}')
print(f'Features avail: {[c for c in FEATURE_COLS if c in panel.columns]}')
print('\nTarget (3d fwd return) summary:')
print(panel['target'].describe().round(4))


Event days: 243
Panel shape   : (3510, 26)
Event dates   : 234
Stocks        : 15
Features avail: ['oil_return_1d', 'oil_return_2d', 'oil_return_5d', 'oil_vol_20d', 'market_ret_1d', 'market_vol_20d', 'sentiment_1d', 'sentiment_5d', 'oil_sent_interact', 'oil_direction', 'vol_regime', 'usdinr_ret_1d', 'usdinr_ret_3d', 'usdinr_ret_5d', 'usdinr_vol_20d', 'oil_fx_interact', 'oil_fx_corr_20d', 'relative_return', 'stock_ret_1d', 'stock_mom_5d', 'stock_mom_20d', 'stock_vol_20d', 'oil_stock_corr_60d', 'stock_beta_nifty_60d', 'stock_idio_vol_20d']

Target (3d fwd return) summary:
count    3510.0000
mean        0.0001
std         0.0254
min        -0.1191
25%        -0.0161
50%         0.0000
75%         0.0163
max         0.1012
Name: target, dtype: float64


In [6]:
# Train/test split (70% train / 30% test, time-ordered)
train, test = time_split(panel, train_ratio=0.70)
print(f'Train: {len(train)} rows, {train.index.get_level_values("date").nunique()} dates')
print(f'Test : {len(test)}  rows, {test.index.get_level_values("date").nunique()} dates')

# Ridge (interpretable baseline)
print('\nTraining Ridge ...')
ridge = AlphaModel(model_type='ridge').fit(train)

# LightGBM (production model)
print('\nTraining LightGBM ...')
lgbm = AlphaModel(model_type='lgbm').fit(train)

# Walk-forward CV comparison
print('\nWalk-Forward CV Results:')
print(f'  Ridge    — IC: {ridge.cv_metrics_["mean_ic"]:+.4f}  hit: {ridge.cv_metrics_["mean_hit"]:.1%}')
print(f'  LightGBM — IC: {lgbm.cv_metrics_["mean_ic"]:+.4f}  hit: {lgbm.cv_metrics_["mean_hit"]:.1%}')


Train: 2445 rows, 163 dates
Test : 1065  rows, 71 dates

Training Ridge ...

 Top Feature Importances:
oil_return_2d: -0.0029
oil_direction: 0.0021
stock_vol_20d: -0.0015
oil_return_1d: 0.0013
stock_mom_5d: -0.001
vol_regime: 0.0008
oil_vol_20d: -0.0008
stock_beta_nifty_60d: -0.0008
stock_idio_vol_20d: 0.0006
market_vol_20d: -0.0004

Training LightGBM ...

 Top Feature Importances:
oil_stock_corr_60d: 224
stock_mom_20d: 209
stock_vol_20d: 201
stock_beta_nifty_60d: 193
stock_mom_5d: 185
stock_ret_1d: 160
stock_idio_vol_20d: 148
relative_return: 127
market_vol_20d: 121
oil_vol_20d: 106

Walk-Forward CV Results:
  Ridge    — IC: -0.0104  hit: 50.9%
  LightGBM — IC: +0.0010  hit: 51.0%


In [7]:
# Feature importance bar chart
fi = lgbm.top_features(15).sort_values()
colors = ['#FF6B35' if 'oil' in k or 'sent' in k else '#8B949E' for k in fi.index]
set_theme()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi.index, fi.values, color=colors, edgecolor='#0D1117', lw=0.4)
ax.set_title('LightGBM Feature Importance (orange = macro/sentiment)')
ax.set_xlabel('Importance')
ax.grid(True, axis='x')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


In [8]:
# OOS predictions and portfolio construction
preds_ridge = ridge.predict(test)
preds_lgbm  = lgbm.predict(test)

# build_portfolio(predicted_returns, actual_returns, top_k, bottom_k, hold_days, tc)
pos_ridge = build_portfolio(preds_ridge, returns, top_k=0.30, bottom_k=0.30, hold_days=3)
pos_lgbm  = build_portfolio(preds_lgbm,  returns, top_k=0.30, bottom_k=0.30, hold_days=3)

print('Sample positions (LightGBM):')
print(pos_lgbm[['date','stock','signal','predicted_return','net_return','hit']].head(10).to_string(index=False))


Sample positions (LightGBM):
      date      stock  signal  predicted_return  net_return   hit
2024-01-08       ONGC       1            0.2099     -0.4317 False
2024-01-08       BPCL       1            0.0374      0.0658  True
2024-01-08       HPCL      -1           -0.6223     -0.0829 False
2024-01-08 ADANIPORTS      -1           -1.3859      0.3870  True
2024-01-08 TATAMOTORS       1            0.1030     -0.4795 False
2024-01-08 ASIANPAINT      -1           -0.5379     -0.2635 False
2024-01-08 HINDUNILVR       1           -0.1457     -0.1744 False
2024-01-08       INFY       1            0.3998     -0.0506 False
2024-01-08      CIPLA      -1           -0.4738      0.1327  True
2024-01-16       ONGC       1            0.5922     -0.9398 False


In [9]:
# Full OOS metrics comparison
all_metrics = {}
for name, positions in [('Ridge', pos_ridge), ('LightGBM', pos_lgbm)]:
    pnl = daily_pnl(positions)
    m   = compute_metrics(pnl)
    ps  = portfolio_summary(positions)
    all_metrics[name] = {**m, **ps}

# NIFTY benchmark on same event days
test_dates = test.index.get_level_values('date').unique()
bench = nifty_benchmark(returns, test_dates)
all_metrics['NIFTY_bench'] = bench

key_metrics = ['sharpe_ann', 'hit_ratio', 'max_drawdown', 'total_return', 'profit_factor', 'cagr']
comp = pd.DataFrame(
    {k: {m: all_metrics[m].get(k, '—') for m in all_metrics} for k in key_metrics}
).T
print('=== OUT-OF-SAMPLE COMPARISON ===')
print(comp.to_string())
save_table(comp.reset_index(), 'model_comparison')


=== OUT-OF-SAMPLE COMPARISON ===
                 Ridge  LightGBM  NIFTY_bench
sharpe_ann     -0.1150   -2.7730       0.9340
hit_ratio       0.4695    0.4491       0.5493
max_drawdown  -12.8200  -22.2530      -5.3870
total_return   -2.3500  -20.0400       3.5700
profit_factor   0.9810    0.6300       1.1690
cagr           -8.1100  -54.7800      13.2700
Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\model_comparison.csv


,index,Ridge,LightGBM,NIFTY_bench
0,sharpe_ann,-0.1150,-2.7730,0.9340
1,hit_ratio,0.4695,0.4491,0.5493
2,max_drawdown,-12.8200,-22.2530,-5.3870
3,total_return,-2.3500,-20.0400,3.5700
4,profit_factor,0.9810,0.6300,1.1690
5,cagr,-8.1100,-54.7800,13.2700


In [10]:
# Information Coefficient (IC) time series
# compute_ic(predictions: pd.Series, actuals: pd.Series) -> pd.Series
actuals_s = test['target']
ic_series = compute_ic(preds_lgbm, actuals_s)
print(f'Mean IC: {ic_series.mean():.4f}  IC>0: {(ic_series>0).mean():.0%}  ICIR: {ic_series.mean()/ic_series.std():.3f}')


Mean IC: -0.0332  IC>0: 48%  ICIR: -0.112


In [11]:
# Equity curve + drawdown + IC bars
set_theme()
fig, axes = plt.subplots(3, 1, figsize=(12, 10), gridspec_kw={'height_ratios': [3, 1, 1]})

for name, positions, color in [('Ridge', pos_ridge, '#8B949E'), ('LightGBM', pos_lgbm, '#58A6FF')]:
    pnl_s = daily_pnl(positions)
    pnls  = pnl_s.values / 100
    eq    = np.cumprod(1 + pnls)
    axes[0].plot(range(len(eq)), eq, color=color, lw=2, label=name)

axes[0].axhline(1, color='#555', lw=0.7)
axes[0].set_ylabel('Portfolio Value')
axes[0].set_title('Cross-Sectional Macro Alpha — OOS Equity Curves')
axes[0].legend(); axes[0].grid(True)

pnls_lgbm = daily_pnl(pos_lgbm).values / 100
eq_lgbm   = np.cumprod(1 + pnls_lgbm)
dd        = (eq_lgbm - np.maximum.accumulate(eq_lgbm)) / np.maximum.accumulate(eq_lgbm)
axes[1].fill_between(range(len(dd)), dd * 100, 0, alpha=0.7, color='#F85149')
axes[1].set_ylabel('DD %'); axes[1].grid(True)

axes[2].bar(range(len(ic_series)), ic_series.values,
            color=['#3FB950' if v > 0 else '#F85149' for v in ic_series.values], lw=0)
axes[2].axhline(0, color='#555', lw=0.7)
axes[2].set_ylabel('IC'); axes[2].set_xlabel('Event Date #'); axes[2].grid(True)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'alpha_system_oos.png', dpi=150, bbox_inches='tight')
plt.show()


In [12]:
# Stock-level PnL attribution
stock_attr = pos_lgbm.groupby('stock').agg(
    mean_net_pnl=('net_return', 'mean'),
    hit_ratio=('hit', 'mean'),
    n_trades=('hit', 'count'),
    mean_signal=('signal', 'mean'),
).round(3).sort_values('mean_net_pnl', ascending=False)
save_table(stock_attr.reset_index(), 'stock_attribution')
print(stock_attr.to_string())


Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\stock_attribution.csv
            mean_net_pnl  hit_ratio  n_trades  mean_signal
stock                                                     
MARUTI             0.145      0.639        36        0.000
ONGC               0.092      0.565        46        0.043
TATAMOTORS         0.064      0.286        56        0.571
HPCL               0.061      0.286        49        0.510
CIPLA              0.050      0.667        33       -0.091
HINDUNILVR         0.009      0.571        35       -0.029
ITC               -0.029      0.432        37       -0.081
IOC               -0.034      0.460        50        0.400
BPCL              -0.042      0.439        41       -0.220
RELIANCE          -0.092      0.429        49        0.388
INFY              -0.107      0.385        39        0.179
TCS               -0.110      0.487        39        0.179
ADANIPORTS        -0.114      0.512        41       -0.171
ASIANPAINT        -0.163      

In [13]:
# Save model + simulate live signal generation
lgbm.save('alpha_model')
print('Model saved.')

# build_live_features(returns, sentiment_series, stocks=None, as_of=None)
live        = build_live_features(returns, sent)
live_preds  = lgbm.predict_latest(live)
live_sigs   = rank_signals(live_preds)
live_weights= signal_weights(live_sigs)

today_df = pd.DataFrame({
    'stock':            live_preds.index.get_level_values('stock'),
    'predicted_return': (live_preds.values * 100).round(3),
    'signal':           live_sigs.values.astype(int),
    'weight':           live_weights.values.round(3),
}).pipe(lambda d: d[d['signal'] != 0]).reset_index(drop=True)

print(f"Today's signals ({returns.index[-1].date()}):")
print(today_df.sort_values('predicted_return', ascending=False).to_string(index=False))
print('NB06 complete ✓')


Model saved.
Today's signals (2026-05-25):
     stock  predicted_return  signal  weight
      INFY             0.761       1   0.125
TATAMOTORS             0.287       1   0.125
      HPCL             0.287       1   0.125
      ONGC             0.039       1   0.125
     CIPLA            -0.467       1   0.125
HINDUNILVR            -0.498       1   0.125
       TCS            -0.671       1   0.125
  RELIANCE            -0.764       1   0.125
    INDIGO            -0.888      -1  -0.143
       ITC            -0.948      -1  -0.143
       IOC            -1.015      -1  -0.143
      BPCL            -1.055      -1  -0.143
ADANIPORTS            -1.229      -1  -0.143
ASIANPAINT            -1.487      -1  -0.143
    MARUTI            -1.509      -1  -0.143
NB06 complete ✓
